# 18 — Extension de la cohorte tabulaire ADNI

**Objectif :** construire une nouvelle cohorte sans imposer la disponibilité d’IRM ou de PET. Aucun fichier existant n’est modifié. Aucun modèle n’est entraîné ici.

Cette expérience est distincte des notebooks 00–17. Leur code récent n’étant pas fourni, ce notebook ne prétend pas reproduire exactement la cible du notebook 17.

**Définition explicite de la nouvelle cible :** diagnostic à une visite proche de deux ans, comparé au diagnostic de départ (CN ou MCI). Une aggravation vaut 1, sinon 0. Il s’agit de l’état observé à cette visite, **pas d’une aggravation survenue à n’importe quel moment du suivi**. Une réversion éventuelle est signalée.

- T0 = premier diagnostic daté, interprétable et non contradictoire du participant.
- Visite cible = diagnostic le plus proche de T0 + 730 jours dans [640, 820] jours. En cas d’égalité, la date la plus ancienne est retenue.
- Aucune visite cible dans cette fenêtre : cible inconnue, participant exclu du tableau de modélisation.
- Variables explicatives : uniquement des mesures datées au plus tard à T0.
- Une ligne par participant. Pas d’imputation, de standardisation ni de sélection statistique ici.

**Exécution :** sélectionner le noyau Python de `E:\Projet_Alzheimer\.venv`, puis exécuter les cellules dans l’ordre. Les sorties affichent des statistiques agrégées. Les CSV individuels restent dans `data/`.

Les effectifs seront calculés sur tes fichiers. Les performances de cette nouvelle cible ne seront pas directement comparables aux AUC de la présentation.

In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import sys
import platform
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path(r"E:\Projet_Alzheimer")
TABLES = ROOT / "data" / "02_Tables_originales"
HORIZON_JOURS = 730
FENETRE_CIBLE = (640, 820)
DELAI_CLINIQUE = 180
DELAI_LCR = 365
DELAI_DEMOGRAPHIE = 365

# Si plusieurs versions non identiques sont présentes, renseigner ici
# le chemin exact souhaité. Ne pas mélanger automatiquement les exports.
FICHIERS_EXPLICITES = {
    # "Diagnostic": TABLES / "All_Subjects_DXSUM_23Oct2026.csv",
}
# Facultatif : CSV exact de l'ancienne cohorte, avec une colonne PTID.
ANCIENNE_COHORTE = None

assert TABLES.is_dir(), f"Dossier absent : {TABLES}"
assert FENETRE_CIBLE[0] <= HORIZON_JOURS <= FENETRE_CIBLE[1]
print("Python :", sys.executable)
print("Pandas :", pd.__version__)
print("Horizon et fenêtre :", HORIZON_JOURS, FENETRE_CIBLE)
print("Aucun import de config_projet : notebook autonome.")

## 1. Choisir et tracer les fichiers sources
Les copies strictement identiques sont reconnues par leur empreinte SHA-256. Des versions différentes entraînent un arrêt : leur choix doit être explicite. Les sources autres que le diagnostic peuvent être absentes ; leurs variables resteront manquantes et cela sera indiqué.

In [ ]:
PREFIXES = {
    "Diagnostic": "All_Subjects_DXSUM_",
    "Demographie": "All_Subjects_PTDEMOG_",
    "MMSE": "All_Subjects_MMSE_",
    "CDR": "All_Subjects_CDR_",
    "ADAS": "All_Subjects_ADAS_",
    "FAQ": "All_Subjects_FAQ_",
    "APOE": "All_Subjects_APOERES_",
    "LCR": "All_Subjects_UPENNBIOMK_ROCHE_",
}

def sha256(p):
    h = hashlib.sha256()
    with p.open("rb") as f:
        for bloc in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

csvs = sorted(p for p in TABLES.rglob("*") if p.is_file() and p.suffix.lower() == ".csv")
sources, lignes_sources, ambigus = {}, [], []
for nom, prefixe in PREFIXES.items():
    if nom in FICHIERS_EXPLICITES:
        candidats = [Path(FICHIERS_EXPLICITES[nom])]
        if not candidats[0].is_file():
            raise FileNotFoundError(candidats[0])
    else:
        candidats = [p for p in csvs if p.name.upper().startswith(prefixe.upper())]
    empreintes = {p: sha256(p) for p in candidats}
    if len(set(empreintes.values())) > 1:
        ambigus.append(nom)
        for p, h in empreintes.items():
            lignes_sources.append(dict(source=nom, fichier=str(p), sha256=h, statut="VERSION À CHOISIR"))
        continue
    choisi = candidats[0] if candidats else None
    sources[nom] = choisi
    lignes_sources.append(dict(source=nom, fichier=str(choisi) if choisi else "",
                              sha256=empreintes.get(choisi, ""),
                              statut="OK" if choisi else "ABSENT",
                              copies_identiques=len(candidats)))
inventaire_sources = pd.DataFrame(lignes_sources)
display(inventaire_sources[["source", "fichier", "statut"]])
if ambigus:
    raise ValueError(f"Renseigner FICHIERS_EXPLICITES en cellule 1 pour : {ambigus}, puis relancer.")
if sources["Diagnostic"] is None:
    raise FileNotFoundError("La table DXSUM est indispensable.")

def lire_source(nom):
    p = sources.get(nom)
    if p is None:
        return None
    t = pd.read_csv(p, dtype="string", encoding="utf-8-sig")
    t.columns = t.columns.str.strip().str.upper()
    if t.columns.duplicated().any():
        raise ValueError(f"Colonnes dupliquées : {nom}")
    if "PTID" not in t:
        raise ValueError(f"Colonne PTID absente : {nom}")
    t["PTID"] = t["PTID"].str.strip().str.upper().replace("", pd.NA)
    return t

def dates_iso(s):
    # Ne pas deviner l'ordre jour/mois. Accepte les dates ISO et leurs horodatages.
    s = s.astype("string").str.strip()
    return pd.to_datetime(s.str.slice(0, 10), format="%Y-%m-%d", errors="coerce").dt.normalize()


## 2. Nettoyer les diagnostics
On accepte les codes 1 (CN), 2 (MCI), 3 (démence) de l’export harmonisé. Le code 10 et les codes absents ne constituent pas une cible. Les conflits d’un même participant à une même date sont exclus. Le premier diagnostic valide est pris avant d’exclure les participants déjà au stade démence : on ne les réintroduit pas à une visite ultérieure.

In [ ]:
dx_brut = lire_source("Diagnostic")
requis = {"PTID", "EXAMDATE", "DIAGNOSIS"}
if not requis.issubset(dx_brut.columns):
    raise ValueError(f"Colonnes manquantes : {requis - set(dx_brut.columns)}")
dx = dx_brut.copy()
dx["date_dx"] = dates_iso(dx["EXAMDATE"])
dx["code_dx"] = pd.to_numeric(dx["DIAGNOSIS"], errors="coerce")
id_ok = dx["PTID"].str.fullmatch(r"\d{3}_S_\d{4,}", na=False)
qualite_dx = pd.DataFrame([
    {"critere": "Lignes brutes", "nombre": len(dx)},
    {"critere": "Identifiants absents ou non reconnus", "nombre": int((~id_ok).sum())},
    {"critere": "Dates absentes ou non reconnues", "nombre": int(dx["date_dx"].isna().sum())},
    {"critere": "Codes autres que 1, 2, 3 ou absents", "nombre": int((~dx["code_dx"].isin([1,2,3])).sum())},
])
dx = dx.loc[id_ok & dx["date_dx"].notna() & dx["code_dx"].isin([1,2,3]),
            ["PTID", "date_dx", "code_dx"]].copy()
nb_codes = dx.groupby(["PTID", "date_dx"])["code_dx"].nunique()
conflits = nb_codes[nb_codes > 1].reset_index()[["PTID", "date_dx"]]
index_conflit = pd.MultiIndex.from_frame(conflits)
masque_conflit = pd.MultiIndex.from_frame(dx[["PTID", "date_dx"]]).isin(index_conflit)
dx = dx.loc[~masque_conflit].drop_duplicates().sort_values(["PTID", "date_dx"])
dx["code_dx"] = dx["code_dx"].astype(int)
display(qualite_dx)
print("Couples participant/date contradictoires exclus :", len(conflits))
print("Participants avec diagnostic utilisable :", dx["PTID"].nunique())
print("Dates diagnostiques utilisables :", len(dx))

## 3. Construire T0 et la cible à environ deux ans
La sélection de la visite cible dépend uniquement de sa date, jamais de son diagnostic. Les participants sans visite dans la fenêtre restent dans l’audit avec une cible inconnue. Les codes intermédiaires ne remplacent pas celui de la visite cible.

In [ ]:
def construire_cibles(diagnostics, horizon=730, fenetre=(640, 820)):
    d = diagnostics.sort_values(["PTID", "date_dx"]).copy()
    depart = d.drop_duplicates("PTID", keep="first").rename(
        columns={"date_dx": "date_T0", "code_dx": "dx_T0"})
    visites = d.merge(depart, on="PTID", validate="many_to_one")
    visites["delai_jours"] = (visites["date_dx"] - visites["date_T0"]).dt.days
    dernier = visites.groupby("PTID")["delai_jours"].max().rename("suivi_observe_jours")
    candidats = visites.loc[visites["delai_jours"].between(*fenetre)].copy()
    candidats["distance_horizon"] = (candidats["delai_jours"] - horizon).abs()
    choisi = (candidats.sort_values(["PTID", "distance_horizon", "date_dx"])
              .drop_duplicates("PTID"))
    choisi = choisi[["PTID", "date_dx", "code_dx", "delai_jours"]].rename(
        columns={"date_dx": "date_cible", "code_dx": "dx_cible"})
    audit = depart.merge(dernier, on="PTID", validate="one_to_one").merge(
        choisi, on="PTID", how="left", validate="one_to_one")
    audit["statut"] = "ELIGIBLE"
    audit.loc[audit["date_cible"].isna(), "statut"] = "SANS_VISITE_DANS_FENETRE"
    audit.loc[audit["dx_T0"].eq(3), "statut"] = "DEMENCE_AU_DEPART"
    audit["cible_aggravation_2ans"] = pd.Series(pd.NA, index=audit.index, dtype="Int64")
    ok = audit["statut"].eq("ELIGIBLE")
    audit.loc[ok, "cible_aggravation_2ans"] = (
        audit.loc[ok, "dx_cible"] > audit.loc[ok, "dx_T0"]).astype(int)
    audit["reversion_a_la_visite_cible"] = (
        audit["dx_cible"] < audit["dx_T0"]).where(ok).astype("boolean")
    return audit

audit_participants = construire_cibles(dx, HORIZON_JOURS, FENETRE_CIBLE)
cohorte = audit_participants.loc[audit_participants["statut"].eq("ELIGIBLE")].copy()
if cohorte.empty:
    raise ValueError("Aucun participant éligible : vérifier les dates et les paramètres.")
assert cohorte["PTID"].is_unique
flux = pd.DataFrame([
    {"etape": "Participants avec diagnostic valide", "participants": len(audit_participants)},
    {"etape": "CN ou MCI au premier diagnostic valide", "participants": int(audit_participants["dx_T0"].isin([1,2]).sum())},
    {"etape": "CN/MCI avec visite dans la fenêtre cible", "participants": len(cohorte)},
])
display(flux)
display(audit_participants["statut"].value_counts().rename_axis("statut").to_frame("participants"))
display(pd.crosstab(cohorte["dx_T0"].map({1:"CN",2:"MCI"}),
                    cohorte["cible_aggravation_2ans"], margins=True))
print("Délai cible min / médian / max :", cohorte["delai_jours"].min(),
      cohorte["delai_jours"].median(), cohorte["delai_jours"].max())
print("Réversions à la visite cible :", int(cohorte["reversion_a_la_visite_cible"].sum()))

## 4. Associer les mesures antérieures à T0
Pour chaque variable, prendre la dernière valeur valide au plus tard à T0, dans la fenêtre indiquée. Les valeurs contradictoires le même jour sont mises de côté. Les dates et l’ancienneté de chaque mesure sont conservées.

Les bornes ci-dessous servent à écarter les codes spéciaux et valeurs impossibles pour les scores. Les résultats de laboratoire non numériques (par exemple « > valeur ») restent manquants : aucune valeur numérique n’est inventée. Leur interprétation peut être affinée plus tard avec le dictionnaire du laboratoire.

L’APOE ne varie pas avec le temps, mais sa date de test est requise ici pour garantir sa disponibilité à T0. Le jour du test sert de proxy de disponibilité, pas de preuve de la date de restitution. Les dates absentes ne sont pas remplacées par celles d’autres visites.

In [ ]:
audit_mesures = []
base = cohorte.copy()
ref = base[["PTID", "date_T0"]].copy()

# bornes : contrôles de plausibilité, pas critères cliniques de diagnostic.
SPEC = [
    ("MMSE", "VISDATE", "MMSCORE", "MMSE_debut", DELAI_CLINIQUE, 0, 30),
    ("CDR", "VISDATE", "CDGLOBAL", "CDR_global", DELAI_CLINIQUE, 0, 3),
    ("CDR", "VISDATE", "CDRSB", "CDR_SB", DELAI_CLINIQUE, 0, 18),
    ("ADAS", "VISDATE", "TOTAL13", "ADAS13", DELAI_CLINIQUE, 0, 85),
    ("FAQ", "VISDATE", "FAQTOTAL", "FAQ_debut", DELAI_CLINIQUE, 0, 30),
    ("LCR", "EXAMDATE", "ABETA42", "ABETA42", DELAI_LCR, 0, None),
    ("LCR", "EXAMDATE", "TAU", "TAU", DELAI_LCR, 0, None),
    ("LCR", "EXAMDATE", "PTAU", "PTAU", DELAI_LCR, 0, None),
    ("Demographie", "VISDATE", "PTGENDER", "sexe", DELAI_DEMOGRAPHIE, 1, 2),
    ("Demographie", "VISDATE", "PTEDUCAT", "education", DELAI_DEMOGRAPHIE, 0, 30),
    ("Demographie", "VISDATE", "PTDOBYY", "annee_naissance", DELAI_DEMOGRAPHIE, 1850, 2026),
]
cache = {nom: lire_source(nom) for nom in PREFIXES if nom != "Diagnostic"}

def associer_mesure(reference, table, col_date, col_valeur, nom, tolerance, minimum=None, maximum=None):
    resultat = reference[["PTID"]].copy()
    resultat[nom] = np.nan
    resultat["date_" + nom] = pd.NaT
    resume = dict(variable=nom, statut="OK", valeurs_non_valides=0,
                  dates_non_valides=0, jours_contradictoires=0)
    if table is None or not {col_date, col_valeur}.issubset(table.columns):
        resume["statut"] = "SOURCE_OU_COLONNE_ABSENTE"
        return resultat, resume
    t = table.loc[table["PTID"].isin(set(reference["PTID"])), ["PTID", col_date, col_valeur]].copy()
    t["date_mesure"] = dates_iso(t[col_date])
    t[nom] = pd.to_numeric(t[col_valeur], errors="coerce")
    valide = t[nom].notna() & np.isfinite(t[nom].astype(float))
    if minimum is not None: valide &= t[nom].ge(minimum).fillna(False)
    if maximum is not None: valide &= t[nom].le(maximum).fillna(False)
    if nom in {"sexe", "annee_naissance"}: valide &= t[nom].mod(1).eq(0).fillna(False)
    if nom == "CDR_global": valide &= t[nom].isin([0, 0.5, 1, 2, 3])
    if nom in {"ABETA42", "TAU", "PTAU"}: valide &= t[nom].gt(0).fillna(False)
    resume["valeurs_non_valides"] = int((~valide).sum())
    resume["dates_non_valides"] = int(t["date_mesure"].isna().sum())
    t = t.loc[valide & t["date_mesure"].notna(), ["PTID", "date_mesure", nom]]
    t[nom] = t[nom].astype(float)
    comptes = t.groupby(["PTID", "date_mesure"])[nom].nunique()
    conflits = comptes[comptes > 1].index
    resume["jours_contradictoires"] = len(conflits)
    t = t.loc[~pd.MultiIndex.from_frame(t[["PTID", "date_mesure"]]).isin(conflits)].drop_duplicates()
    if t.empty: return resultat, resume
    resultat = pd.merge_asof(
        reference.sort_values("date_T0"), t.sort_values("date_mesure"),
        left_on="date_T0", right_on="date_mesure", by="PTID",
        direction="backward", tolerance=pd.Timedelta(days=tolerance) if tolerance is not None else None)
    resultat = resultat[["PTID", nom, "date_mesure"]].rename(columns={"date_mesure": "date_"+nom})
    return resultat, resume

for source, date, champ, nom, tol, vmin, vmax in SPEC:
    valeurs, resume = associer_mesure(ref, cache[source], date, champ, nom, tol, vmin, vmax)
    base = base.merge(valeurs, on="PTID", how="left", validate="one_to_one")
    audit_mesures.append(resume)

apoe = cache["APOE"]
if apoe is not None and "GENOTYPE" in apoe:
    apoe = apoe.copy()
    geno = apoe["GENOTYPE"].str.replace(" ", "", regex=False)
    valide = geno.str.fullmatch(r"[234]/[234]", na=False)
    apoe["NB_E4"] = geno.where(valide).map(
        lambda s: str(s).split("/").count("4") if pd.notna(s) else np.nan)
valeurs, resume = associer_mesure(ref, apoe, "APTESTDT", "NB_E4", "APOE_e4", None, 0, 2)
base = base.merge(valeurs, on="PTID", how="left", validate="one_to_one")
audit_mesures.append(resume)

# Âge approximatif : année de T0 moins année de naissance connue à T0.
base["age_reference"] = base["date_T0"].dt.year - base["annee_naissance"]
base.loc[~base["age_reference"].between(40, 120), "age_reference"] = np.nan
base["date_age_reference"] = base["date_annee_naissance"]

FEATURES = ["MMSE_debut", "CDR_global", "CDR_SB", "ADAS13", "FAQ_debut",
            "ABETA42", "TAU", "PTAU", "education", "sexe", "age_reference", "APOE_e4"]
for nom in FEATURES:
    base["anciennete_" + nom + "_jours"] = (base["date_T0"] - base["date_" + nom]).dt.days
print("Variables assemblées :", len(FEATURES))
display(pd.DataFrame(audit_mesures))

## 5. Vérifications et couverture
La cohorte finale exige au moins une des 12 variables présente. Cela ne garantit pas qu’un profil très incomplet soit exploitable : le nombre de variables présentes est audité. On ne supprime pas automatiquement les participants sans LCR et on ne remplit aucune valeur manquante avant la séparation entraînement/test.

Les dates, diagnostics futurs, durée de suivi et cible sont des colonnes d’audit : elles ne font pas partie de `FEATURES`. Le sexe devra être traité comme une catégorie dans le futur pipeline.

In [ ]:
assert base["PTID"].is_unique
assert base["dx_T0"].isin([1,2]).all()
assert base["delai_jours"].between(*FENETRE_CIBLE).all()
assert base["cible_aggravation_2ans"].isin([0,1]).all()
for nom in FEATURES:
    present = base[nom].notna()
    assert base.loc[present, "date_"+nom].notna().all(), f"Date absente : {nom}"
    assert (base.loc[present, "date_"+nom] <= base.loc[present, "date_T0"]).all(), f"Mesure future : {nom}"

base["nb_variables_presentes"] = base[FEATURES].notna().sum(axis=1)
base["exploitable_tabulaire"] = base["nb_variables_presentes"].gt(0)
final = base.loc[base["exploitable_tabulaire"]].copy()
if final.empty: raise ValueError("Aucune variable présente : contrôler les tables et leurs dates.")
couverture = pd.DataFrame({
    "variable": FEATURES,
    "presents": [int(final[c].notna().sum()) for c in FEATURES],
    "manquants": [int(final[c].isna().sum()) for c in FEATURES],
})
couverture["manquants_pct"] = (100 * couverture["manquants"] / len(final)).round(1)
display(couverture)
display(base["nb_variables_presentes"].value_counts().sort_index().rename_axis(
    "nb_variables_presentes").to_frame("participants"))
print("Participants éligibles à la cible :", len(base))
print("Sans aucune variable présente :", int((~base["exploitable_tabulaire"]).sum()))
print("Participants dans le nouveau tableau :", len(final))
print("Aggravations à la visite cible :", int(final["cible_aggravation_2ans"].sum()))
print("Variables entièrement absentes :", couverture.loc[couverture["presents"].eq(0), "variable"].tolist())
print("Les effectifs ne sont pas directement comparables aux 312 : nouvelle date de départ et cible explicite.")

## 6. Comparaison facultative avec les anciens participants
Renseigner `ANCIENNE_COHORTE` en première cellule pour mesurer le chevauchement. On ne recherche pas automatiquement un fichier ambigu. Un participant déjà utilisé pour choisir le modèle ne constitue pas une nouvelle observation indépendante de validation.

In [ ]:
comparaison_ancienne = pd.DataFrame()
if ANCIENNE_COHORTE is None:
    print("Comparaison non effectuée : chemin ANCIENNE_COHORTE non renseigné.")
else:
    ancien = pd.read_csv(Path(ANCIENNE_COHORTE), dtype="string", encoding="utf-8-sig")
    if "PTID" not in ancien: raise ValueError("L'ancien CSV doit contenir PTID.")
    anciens = set(ancien["PTID"].str.strip().str.upper().dropna())
    nouveaux = set(final["PTID"])
    final["present_ancienne_cohorte"] = final["PTID"].isin(anciens)
    comparaison_ancienne = pd.DataFrame([
        {"groupe": "Ancienne cohorte fournie", "participants": len(anciens)},
        {"groupe": "Nouvelle cohorte", "participants": len(nouveaux)},
        {"groupe": "En commun", "participants": len(anciens & nouveaux)},
        {"groupe": "Participants supplémentaires", "participants": len(nouveaux - anciens)},
        {"groupe": "Anciens non retenus avec ces critères", "participants": len(anciens - nouveaux)},
    ])
    display(comparaison_ancienne)

## 7. Export dans un nouveau dossier
Un dossier horodaté distinct est créé à chaque exécution de cette cellule. Les tables originales, notebooks précédents et anciens résultats restent intacts. Le tableau `variables_X.csv` contient seulement PTID et les variables explicatives ; `PTID` sert à faire les jointures, jamais à prédire.

In [ ]:
SORTIE = ROOT / "data" / "05_Manifestes" / (
    "extension_tabulaire_18_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
SORTIE.mkdir(parents=True, exist_ok=False)

def sauver(df, nom):
    df.to_csv(SORTIE / nom, index=False, encoding="utf-8-sig")

sauver(inventaire_sources, "sources.csv")
sauver(qualite_dx, "audit_diagnostics.csv")
sauver(conflits, "conflits_diagnostics.csv")
sauver(audit_participants, "audit_eligibilite_tous_participants.csv")
sauver(base, "cohorte_eligible_avant_exclusion_profils_vides.csv")
sauver(flux, "effectifs_selection.csv")
sauver(pd.DataFrame(audit_mesures), "audit_mesures.csv")
sauver(couverture, "couverture_variables.csv")
sauver(final, "cohorte_tabulaire_elargie.csv")
sauver(final[["PTID"] + FEATURES], "variables_X.csv")
sauver(final[["PTID", "cible_aggravation_2ans"]], "cible_y.csv")
if not comparaison_ancienne.empty:
    sauver(comparaison_ancienne, "comparaison_ancienne_cohorte.csv")

config = {
    "experience": "18_extension_tabulaire_diagnostic_vers_2ans",
    "date_creation": datetime.now().isoformat(),
    "T0": "premier diagnostic valide et non contradictoire",
    "population": "CN ou MCI à T0 ; aucune obligation IRM ou PET",
    "cible": "diagnostic à la visite la plus proche de 730 jours supérieur au diagnostic T0",
    "horizon_jours": HORIZON_JOURS,
    "fenetre_jours": list(FENETRE_CIBLE),
    "egalite_dates": "date la plus ancienne",
    "evenement_intermediaire": "non utilisé pour remplacer le diagnostic de la visite cible",
    "sans_visite_cible": "exclu, cible inconnue",
    "delai_clinique_jours": DELAI_CLINIQUE,
    "delai_lcr_jours": DELAI_LCR,
    "delai_demographie_jours": DELAI_DEMOGRAPHIE,
    "apoe": "dernier test valide au plus tard à T0, sans délai maximal",
    "age": "approximation par année ; contrôle 40–120 ans",
    "variables": FEATURES,
    "categorielle": ["sexe"],
    "imputation": "aucune ; à apprendre dans les plis d'entraînement",
    "participants": len(final),
    "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
    "ancienne_cohorte": str(ANCIENNE_COHORTE) if ANCIENNE_COHORTE else None,
    "comparabilite": "ne reproduit pas automatiquement la cible ou la population du notebook 17",
}
(SORTIE / "protocole.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
(SORTIE / "LIRE_AVANT_MODELISATION.txt").write_text(
    "Nouvelle expérience : diagnostic à environ deux ans, pas aggravation sur tout le suivi.\n"
    "Utiliser exclusivement les variables listées dans protocole.json. PTID n'est pas une entrée.\n"
    "Conserver les mêmes participants de validation pour comparer les tailles d'entraînement.\n"
    "Ne pas utiliser des participants déjà vus pour choisir le modèle comme test totalement indépendant.\n"
    "Imputation, encodage, standardisation, calibration et réglages dans l'entraînement uniquement.\n"
    "Séparer par participant. Choisir le seuil sans consulter le test final.\n"
    "Aucune supériorité de performance n'est garantie par l'extension.\n", encoding="utf-8")

print("EXPORT TERMINÉ")
print("Participants :", len(final))
print("Cible 0 :", int(final["cible_aggravation_2ans"].eq(0).sum()))
print("Cible 1 :", int(final["cible_aggravation_2ans"].eq(1).sum()))
print("Dossier :", SORTIE)
print("Envoie le bilan et le tableau de couverture avant l'entraînement.")